# Teaching a 7B Model to Beat GPT-4o at Chess

In this notebook, we'll train **Qwen 2.5 7B Instruct** to play chess using:
1. **Supervised Fine-Tuning (SFT)** — teach the model chess move format and basic strategy
2. **GRPO (Group Relative Policy Optimization)** — reinforcement learning with Stockfish as the reward signal

All heavy compute runs on UVA's Rivanna HPC cluster via the [`rv` CLI](https://rivanna.dev). This notebook runs locally and dispatches jobs.

**Prerequisites:**
- `rv` CLI installed and configured (`rv init`)
- VPN connected to UVA Anywhere
- `.env` file with `OPENAI_API_KEY`, `HF_TOKEN`, `WANDB_API_KEY`

## 1. Setup & Prerequisites

In [ ]:
# Install local dependencies (python-chess for interactive cells)
!uv pip install python-chess openai

In [ ]:
# Check rv is installed and connected
!rv status

In [ ]:
# Push API keys to Rivanna (these are injected into every job)
# ensure that you've copied the .env file and made it yourself; run `cp .env.example .env` and fill in your keys first!!
!rv env import .env

In [ ]:
# Download Stockfish chess engine to Rivanna
!rv run --mig bash scripts/setup_stockfish.sh

In [ ]:
# Validate that dependencies install correctly on the cluster
!rv run --mig python -c "import trl, chess, wandb, peft, datasets; print('All imports OK')"

## 2. Understanding the Chess Environment

Before training, let's understand how we represent chess positions and moves.
We use:
- **FEN** (Forsyth-Edwards Notation) — compact string representation of a board position
- **UCI** (Universal Chess Interface) — move format like `e2e4` (from-square to-square)

In [ ]:
import chess

# Create the starting position
board = chess.Board()
print("Starting position FEN:")
print(board.fen())
print()
print(board)

In [ ]:
# Legal moves in UCI format — this is what we'll feed to the model
legal_moves = [m.uci() for m in board.legal_moves]
print(f"{len(legal_moves)} legal moves: {' '.join(legal_moves)}")

In [ ]:
# Make some moves and see how the board changes
board.push(chess.Move.from_uci("e2e4"))  # King's pawn
board.push(chess.Move.from_uci("e7e5"))  # Symmetric response
board.push(chess.Move.from_uci("g1f3"))  # Knight out
print(board)
print(f"\nFEN: {board.fen()}")
print(f"Legal moves for Black: {' '.join(m.uci() for m in board.legal_moves)}")

In [ ]:
# Validate moves — this is how the reward function checks legality
test_board = chess.Board(board.fen())

good_move = chess.Move.from_uci("b8c6")  # Knight to c6 (legal)
bad_move = chess.Move.from_uci("a1a2")   # Rook move (illegal for Black)

print(f"b8c6 is legal: {good_move in test_board.legal_moves}")
print(f"a1a2 is legal: {bad_move in test_board.legal_moves}")

## 3. Building the Reward Function

The reward function is the **heart of GRPO training**. It scores each model output on a progressive scale:

| Component | Weight | What it checks |
|-----------|--------|----------------|
| `format_reward` | 0.1 | Did the model use `<move>` tags? |
| `legality_reward` | 0.4 | Is the move legal in this position? |
| `quality_reward` | 0.5 | How good is the move (Stockfish eval)? |

This graduated approach is critical — the model starts knowing nothing about chess.
With binary rewards, it would get zero signal. With progressive rewards,
even outputting the right format gets *some* reward, enabling learning.

In [ ]:
import re

def extract_move(text):
    """Extract UCI move from <move>...</move> tags."""
    match = re.search(r"<move>\s*([a-h][1-8][a-h][1-8][qrbn]?)\s*</move>", text)
    return match.group(1) if match else None

# Test with various outputs
test_outputs = [
    "I think the best move is <move>e2e4</move>",        # Good format
    "The move is e2e4",                                   # Missing tags
    "<move>xyz123</move>",                                 # Bad UCI format
    "<move>a1a2</move>",                                   # Valid UCI, may be illegal
]

for output in test_outputs:
    move = extract_move(output)
    print(f"  {output[:50]:50s} -> {move}")

In [ ]:
# Simulate the full reward pipeline on a position
import math

test_fen = "rnbqkbnr/pppppppp/8/8/4P3/8/PPPP1PPP/RNBQKBNR b KQkq - 0 1"
test_board = chess.Board(test_fen)
print(f"Position: {test_fen}")
print(f"Legal moves: {' '.join(m.uci() for m in test_board.legal_moves)}")
print()

test_completions = [
    "<move>e7e5</move>",          # Legal, common response
    "<move>a7a1</move>",          # Valid UCI, but illegal
    "I play e7e5",                # Wrong format
    "<move>d7d5</move>",          # Legal, Scandinavian Defense
]

for comp in test_completions:
    move_str = extract_move(comp)
    fmt = 1.0 if move_str else 0.0
    legal = 0.0
    quality = 0.0
    if move_str:
        try:
            move = chess.Move.from_uci(move_str)
            legal = 1.0 if move in test_board.legal_moves else 0.0
        except ValueError:
            pass
    total = 0.1 * fmt + 0.4 * legal + 0.5 * quality
    print(f"  {comp:30s}  fmt={fmt:.0f}  legal={legal:.0f}  quality={quality:.1f}  total={total:.2f}")

## 4. Preparing Training Data

We use the [Lichess chess puzzles dataset](https://huggingface.co/datasets/Lichess/chess-puzzles) — ~4.5M puzzles with verified best moves from Stockfish analysis.

The data prep script:
1. Loads puzzles rated 1200-2200 (intermediate difficulty)
2. Extracts the puzzle position (after opponent's setup move)
3. Creates **15K examples for SFT** (FEN + best move pairs)
4. Creates **5K examples for GRPO** (FEN positions only — the model discovers good moves via RL)

In [ ]:
# Run data preparation on a free MIG slice (no GPU needed, just CPU + network)
!rv run --mig python scripts/data_prep.py

In [ ]:
# Check the job status
!rv ps

In [ ]:
# View logs
!rv logs

## 5. SFT Warmstart

SFT (Supervised Fine-Tuning) teaches the model **how** to play chess — specifically:
- The prompt format (FEN + legal moves)
- The output format (`<move>...</move>` tags)
- Basic move selection from high-quality games

We use **LoRA** (Low-Rank Adaptation) so we only train ~0.5% of the model's parameters.
This makes training fast and memory-efficient.

**Why SFT first?** RL can amplify existing capabilities but can't teach them from scratch.
Without SFT, the base model makes legal moves ~5% of the time. After SFT, it should reach >90%.

In [ ]:
# Submit SFT training job (1x A100 80GB, ~1-2 hours)
!rv run -g 1 -t a100 --time 2h --name chess-sft python scripts/sft_train.py

In [ ]:
# Monitor the job
!rv ps

In [ ]:
# Follow training logs (Ctrl+C to stop following)
!rv logs chess-sft -f

In [ ]:
# Check GPU utilization
!rv gpu chess-sft

📊 **Check your W&B dashboard** for training metrics — look for the `chess-sft` run.
Key metrics to watch: `train/loss` should decrease steadily.

## 6. GRPO Training

Now the fun part — **reinforcement learning**. GRPO works like this:

1. For each position, generate **8 candidate moves** (the "group")
2. Score each move with our reward function (format + legality + Stockfish quality)
3. Compute **relative advantages** within the group (better moves get positive advantage)
4. Update the model to make high-advantage moves more likely

Unlike PPO, GRPO doesn't need a separate critic network — it uses the group statistics
as a baseline. This makes it simpler and more memory-efficient.

**Architecture:** 2x A100 GPUs — one runs a **vLLM server** for fast generation,
the other runs training. The wrapper script `run_grpo.sh` manages both processes.

In [ ]:
# Submit GRPO training job (2x A100 80GB, ~4-8 hours)
!rv run -g 2 -t a100 --time 8h --name chess-grpo --single-node bash scripts/run_grpo.sh

In [ ]:
# Monitor
!rv ps

In [ ]:
# Follow GRPO training logs
!rv logs chess-grpo -f

In [ ]:
# Check GPU utilization on both GPUs
!rv gpu chess-grpo

📊 **W&B metrics to watch** (`chess-grpo` run):
- `reward/format` — should approach 1.0 quickly (model learns the tags)
- `reward/legality` — should climb steadily (model makes fewer illegal moves)
- `reward/quality` — slower improvement (move quality from Stockfish)
- `reward/total` — combined signal, should trend upward

## 7. Evaluation — Can We Beat GPT-4o?

Time to test our trained model:
1. **Puzzle accuracy** — legal move rate and Stockfish evaluation on held-out positions
2. **Full games vs GPT-4o** — play actual chess games and track wins/losses

In [ ]:
# Submit evaluation job
!rv run -g 1 -t a100 --time 1h --name chess-eval python scripts/evaluate.py --num_puzzles 100 --num_games 10

In [ ]:
# Follow eval logs
!rv logs chess-eval -f

In [ ]:
# Pull evaluation results to view locally
!rv exec "cat $(ls -t /scratch/$USER/.rv/outputs/chess-eval-*/eval_results.json 2>/dev/null | head -1)"

In [ ]:
import json

# Display results nicely (after pulling)
# You can also run: rv sync pull /scratch/$USER/.rv/outputs/chess-eval-*/eval_results.json ./results/

print("Check rv logs for full results, or pull the JSON:")
print("  rv sync pull /scratch/$USER/.rv/outputs/ ./results/")

## What's Next?

**If you want to improve the model further:**
- Increase SFT data size (we used 15K, you could go to 50K+)
- Run GRPO for more steps (we did 500, try 2000+)
- Add chain-of-thought reasoning traces to SFT data
- Use a higher Stockfish depth in the reward function (depth 20+)

**Key takeaway:** RL can amplify a model's existing chess knowledge, but SFT warmstart
is essential. The progressive reward function (format → legality → quality) is what makes
learning possible — without it, the model gets no useful signal.

**Realistic expectations:** Based on Chess-R1 research, models plateau at ~25-30% puzzle
accuracy. But that's enough to beat GPT-4o, which makes illegal moves ~13% of the time
and plays at roughly 1540 Elo.